# 🌊 River Turbidity Detection - Model Evaluation & Inference

**This Notebook**:
- Evaluate model on test set
- Generate confusion matrix
- Plot ROC curve and AUC
- Calculate detailed metrics
- Make predictions on new images

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import tensorflow as tf
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, f1_score, accuracy_score
)
import json
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

## Step 2: Load Model and Data

In [ ]:
# Load model
MODEL_PATH = Path('./models/xception_final.keras')
model = tf.keras.models.load_model(MODEL_PATH)
print(f"✅ Model loaded from: {MODEL_PATH}")

# Setup data path
PROCESSED_DIR = Path('./data_processed')
TARGET_SIZE = (299, 299)
BATCH_SIZE = 32

# Create test data generator
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    PROCESSED_DIR / 'test',
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"✅ Test data loaded: {test_generator.samples} samples")
print(f"   Classes: {test_generator.class_indices}")

## Step 3: Make Predictions on Test Set

In [ ]:
print("🔮 Making predictions on test set...")
print("   (This may take a few minutes)\n")

# Get predictions
y_pred_proba = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_generator.classes

print(f"\n✅ Predictions completed!")
print(f"   Total predictions: {len(y_pred)}")

## Step 4: Calculate Metrics

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
report = classification_report(y_true, y_pred, 
                               target_names=['Keruh (Turbid)', 'Jernih (Clear)'],
                               output_dict=True)

print("📊 CLASSIFICATION METRICS\n")
print(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\nDetailed Report:")
print("─" * 70)
print(f"{'Class':<20} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("─" * 70)

for class_name, metrics in report.items():
    if class_name not in ['accuracy', 'macro avg', 'weighted avg']:
        print(f"{class_name:<20} {metrics['precision']:<15.4f} {metrics['recall']:<15.4f} {metrics['f1-score']:<15.4f}")

print("─" * 70)
print(f"{'Macro Average':<20} {report['macro avg']['precision']:<15.4f} {report['macro avg']['recall']:<15.4f} {report['macro avg']['f1-score']:<15.4f}")
print(f"{'Weighted Average':<20} {report['weighted avg']['precision']:<15.4f} {report['weighted avg']['recall']:<15.4f} {report['weighted avg']['f1-score']:<15.4f}")

## Step 5: Generate Confusion Matrix

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Keruh', 'Jernih'],
            yticklabels=['Keruh', 'Jernih'],
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save
log_dir = Path('./logs')
log_dir.mkdir(exist_ok=True)
plt.savefig(log_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved!")
print(f"\nConfusion Matrix:")
print(f"                Predicted")
print(f"                Keruh  Jernih")
print(f"Actual Keruh  {cm[0,0]:>6} {cm[0,1]:>6}")
print(f"       Jernih {cm[1,0]:>6} {cm[1,1]:>6}")

## Step 6: Plot ROC Curve

In [ ]:
# Calculate ROC curve
fpr, tpr, _ = roc_curve(y_true, y_pred_proba[:, 1])
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
plt.title('ROC Curve - Test Set', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save
plt.savefig(log_dir / 'roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ ROC curve saved!")
print(f"   ROC AUC Score: {roc_auc:.4f}")

## Step 7: Save Evaluation Results

In [ ]:
# Prepare results dictionary
results = {
    'overall_accuracy': float(accuracy),
    'roc_auc': float(roc_auc),
    'confusion_matrix': cm.tolist(),
    'test_samples': int(test_generator.samples),
    'class_metrics': {}
}

# Add class-specific metrics
class_names = ['Keruh (Turbid)', 'Jernih (Clear)']
for i, class_name in enumerate(class_names):
    results['class_metrics'][class_name] = {
        'precision': float(report[class_name]['precision']),
        'recall': float(report[class_name]['recall']),
        'f1-score': float(report[class_name]['f1-score']),
        'support': int(report[class_name]['support'])
    }

# Save as JSON
results_path = log_dir / 'evaluation_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Results saved to: {results_path}")

## Step 8: Make Predictions on New Images

## Step 9: Example Predictions

In [ ]:
# Make predictions on test images
test_dir = Path('./data_processed/test/keruh')
test_images = list(test_dir.glob('*.jpg'))[:3]

print("🔮 Sample Predictions:\n")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, img_path in enumerate(test_images):
    # Make prediction
    result = predictor.predict_single(img_path)
    
    # Display image
    img = Image.open(img_path)
    axes[idx].imshow(img)
    
    # Add title with prediction
    title = f"{result['class']}\nConfidence: {result['confidence']:.2%}"
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].axis('off')
    
    # Print result
    print(f"Image {idx+1}: {img_path.name}")
    print(f"  Prediction: {result['class']}")
    print(f"  Confidence: {result['confidence']:.2%}")
    print(f"  Probabilities:")
    print(f"    - Keruh: {result['probabilities']['Keruh']:.4f}")
    print(f"    - Jernih: {result['probabilities']['Jernih']:.4f}\n")

plt.tight_layout()
plt.show()

## Step 10: Prediction on Custom Image

## ✨ Summary

✅ Completed:
- Loaded trained model
- Evaluated on test set
- Generated confusion matrix
- Plotted ROC curve
- Calculated comprehensive metrics
- Created reusable predictor class
- Made predictions on sample images

📊 **Key Results**:
- Overall Accuracy: See above
- ROC AUC: See above
- Both visualizations and JSON results saved

🚀 **Next Steps**:
1. Deploy model for real-time predictions
2. Create web interface or API
3. Monitor model performance on new data
4. Retrain if needed with new data